# Week 3, day 1 (afternoon) — Extra practice 11 SOLUTIONS: reading files   (L03)

Executed in the lab image against the real files in `data/`. Every quoted
number is what it actually printed.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Extra practice 11 — Reading files. Run this once.
import pandas as pd

EXPECTED_COLUMNS = ["OrderID", "OrderDate", "Region", "Category", "Product",
                    "ShipMode", "Quantity", "Sales", "Discount", "Profit"]
EXPECTED_ROWS = 300

print("we expect %d rows and %d columns" % (EXPECTED_ROWS, len(EXPECTED_COLUMNS)))

### Question 1

`(300, 10)` -> `check` passes.

Two assertions, one line each: the row count is what you expected, and the
columns are exactly the ones you expected in the order you expected.

That is a load test. It is worth writing before you write any analysis,
because every number you compute afterwards depends on it and none of them
will tell you when it stops holding.

In [ ]:
def check(df):
    return len(df) == EXPECTED_ROWS and list(df.columns) == EXPECTED_COLUMNS

clean = pd.read_csv("../data/sales.csv")
print("rows:", len(clean), "columns:", len(clean.columns))
print("check passes:", check(clean))

### Question 2

`skiprows=2` **False** · `3` **True** · `4` **True** · `5` **False**.

Worksheet 11 Q4 turned into a test, and the test is what makes the
difference. Reading the four outputs by eye, `skiprows=5` gives `(299, 10)`
with plausible-looking data and most people would accept it.

The check catches it because it knows the answer should be 300. That is the
only reason it works — a test that only asserts 'it loaded without error'
would pass all four.

The useful habit is to write down the expected shape *before* you load, from
the source system or the file's own documentation, and assert against it.
Deriving the expectation from the load you just did proves nothing.

In [ ]:
def check(df):
    return len(df) == EXPECTED_ROWS and list(df.columns) == EXPECTED_COLUMNS

for n in (2, 3, 4, 5):
    d = pd.read_csv("../data/sales_report.csv", skiprows=n)
    print("skiprows=%d -> %-11s check: %s" % (n, str(d.shape), check(d)))

### Question 3

`.equals()` is `True`. -> `sales.csv` `33314` bytes, `sales.tsv` `33228` — a difference of **86**.

Identical data, different sizes. The 86 bytes are quotation marks: fields
containing a comma have to be wrapped in the CSV and do not in the TSV,
because no product name contains a tab.

That is the entire argument for TSV, and 86 bytes on 33 KB shows it is a
small effect *for this data*. The reason to care is not size, it is that
quoting is where CSV parsers go wrong — as worksheet 11 Q3 showed, reading
the TSV with the wrong separator raised precisely because of those commas.

In [ ]:
import os
csv = pd.read_csv("../data/sales.csv")
tsv = pd.read_csv("../data/sales.tsv", sep="\t")
print("identical data:", csv.equals(tsv))
print()
print("sales.csv bytes:", os.path.getsize("../data/sales.csv"))
print("sales.tsv bytes:", os.path.getsize("../data/sales.tsv"))
print("difference:     ", os.path.getsize("../data/sales.csv") - os.path.getsize("../data/sales.tsv"))

# The CSV is larger because product names contain commas, so those fields
# have to be wrapped in quotes. The TSV needs no quoting -- no name contains
# a tab.

### Question 4

`63` of 300 rows have a comma in `Product`, across **`24` distinct products**. -> both round-trip identically between the two files.

Two of them are `'"While you Were Out" Message Book, One Form per Page'`
and `'#10 White Business Envelopes,4 1/8 x 9 1/2'` — one carries embedded
double quotes as well as a comma, and the other has no space after its
comma, which is exactly the shape that makes naive splitting look like it
worked.

The values survive both formats intact, which is the point: a correctly
quoted CSV is not lossy. It is only lossy when someone parses it by
splitting on commas.

In [ ]:
csv = pd.read_csv("../data/sales.csv")
tsv = pd.read_csv("../data/sales.tsv", sep="\t")

with_comma = csv["Product"].str.contains(",")
print("rows whose Product contains a comma:", with_comma.sum(), "of", len(csv))
print("distinct such products:", csv.loc[with_comma, "Product"].nunique())
print()
for v in sorted(csv.loc[with_comma, "Product"].unique())[:2]:
    print("  ", repr(v))
print()
i = with_comma.idxmax()
print("same value in both files:", csv.loc[i, "Product"] == tsv.loc[i, "Product"])

### Question 5

`Region` 28 missing (9.3%), `Discount` 43 (14.3%), `Profit` 24 (8.0%). -> **215 of 300** rows are completely clean.

Three columns damaged and 28% of rows affected by at least one of them.

Percentages matter more than counts when you are deciding what to do.
14% missing in `Discount` is a column you can still use with care; if it
were 90% you would drop it. And 215 complete rows means `dropna()` costs
you more than a quarter of the file — a decision, not a cleanup step.

In [ ]:
messy = pd.read_csv("../data/sales_messy.csv", na_values=["?", "Missing"])
missing = messy.isna().sum()
missing = missing[missing > 0]

for col, n in missing.items():
    print("%-10s %3d missing  (%.1f%%)" % (col, n, 100 * n / len(messy)))
print()
print("rows with no missing values at all:", len(messy.dropna()), "of", len(messy))

### Question 6

Only one column differs: **`Discount` — `float64` when clean, `str` when loaded without `na_values`**. -> no exception was raised.

One `?` in a column of numbers turns the whole column to text. The dtype
comparison finds it instantly.

`Region` does not appear because it is text in both cases — `Missing` is
just another string among region names, so the dtype is unchanged and this
check cannot see it. `Profit` does not appear either, because its
placeholder was a genuinely empty field, which `read_csv` already treats as
`NaN`.

So a dtype check catches the most dangerous case — a numeric column
silently becoming text — and misses corruption inside a text column
entirely. It is a good check and not a sufficient one. Pair it with the
value-level look from Q5 and worksheet 07's `value_counts()`.

In [ ]:
clean = pd.read_csv("../data/sales.csv")
bad = pd.read_csv("../data/sales_messy.csv")

for col in clean.columns:
    if clean[col].dtype != bad[col].dtype:
        print("%-10s clean=%-10s messy=%s" % (col, clean[col].dtype, bad[col].dtype))
print()
print("no exception was raised by the bad load")

### Question 7

`(79, 10)` out and `(79, 10)` back, columns match. -> `Sales` total `64684.07` both ways.

A clean round trip: same shape, same columns, same total to the penny.

`index=False` on the way out is what keeps the columns matching — without
it you would read back eleven columns, the extra one called `Unnamed: 0`,
as in worksheet 14 Q5.

Comparing the rounded totals is the right test. Comparing the raw float
sums would be testing something you have no reason to expect, for the
reasons in worksheet 12.

In [ ]:
sales = pd.read_csv("../data/sales.csv", parse_dates=["OrderDate"])
y2012 = sales[sales["OrderDate"].dt.year == 2012]
y2012.to_csv("/tmp/y2012.csv", index=False)

back = pd.read_csv("/tmp/y2012.csv", parse_dates=["OrderDate"])
print("written:", y2012.shape, "read back:", back.shape)
print("columns match:", list(y2012.columns) == list(back.columns))
print()
print("total out:", round(y2012["Sales"].sum(), 2))
print("total in: ", round(back["Sales"].sum(), 2))

### Question 8

`usecols=["OrderID", "Revenue"]` -> **raises** `ValueError: Usecols do not match columns, columns expected but not found: ['Revenue']`.

`usecols` validates against the header and names the column it could not
find. That is the behaviour you want: asking for a column that does not
exist is a mistake, and mistakes should stop you.

Put it next to worksheet 06 Q9, where `drop(columns=[...], errors="ignore")`
quietly did nothing for the same class of error. Two APIs, two philosophies,
and the strict one is the one that will save you.

It is also a free schema check: `usecols` with the columns you require will
fail loudly the day upstream renames one, instead of handing you a frame
with a column silently missing.

In [ ]:
print(pd.read_csv("../data/sales.csv", usecols=["OrderID", "Revenue"]))